# A1 and A2 aeration airflow: compare preparation strategies and predict one hour ahead

Read the shared minute-level `data_filled.csv` and Boolean mask exported by notebook 01 or 02. Recover observations by masking filled cells. Do not read old 20-minute exports or fetch extra SQL history. Source coverage is April 14–23, 2026. Seven-day final training covers April 14–20 and ends before April 21; test origins cover April 21–23.

Compare 24 preparations: raw / unlimited forward fill / five-minute age-limited inputs; 1/5/10/20-minute input summaries; current features / compact 1/3/6-hour lags; a fixed seven-day final training window. Select preparation and its screening algorithm (mean, Ridge or Extra Trees) by mean RMSE on three purged April 18–20 validation days. Additional fixed XGBoost and Random Forest models are comparison references. No strategy is selected on test scores.

All strategies use identical **20-minute forecast origins** and the same **observed exact-minute target one hour later**. Input aggregation never changes the forecasting objective. Missing inputs use training-only median imputation with indicators; unobserved/constant sensors are excluded per fold. Measurement age and observed support accompany values. A 20-minute origin schedule differs from the earlier minute-origin experiments, so compare strategies using this run's common labels.

Long repeated signals are audited and evaluated in a separate sensitivity report; they are not automatically deleted. Earlier source fills cannot be reconstructed if the source mask marked them observed. Validation folds expand over four, five and six days inside the training week; the final fit uses all seven days. Fourteen- and twenty-eight-day histories remain unavailable. These previously inspected test dates remain exploratory.

The primary algorithm comparison is **XGBoost, LSTM and Ridge Regression**, plus persistence. All three use identical 12-step historical sequences sampled every 20 minutes; tree/linear inputs are flattened. LSTM epochs are selected on a purged chronological training-only tail, then the network is refitted on all eligible training sequences. Initial sequence warmup reduces the effective training-origin count; calendar coverage remains seven days.

Forecast, error-comparison and LSTM-loss charts are saved under each target’s `comparison_xgb_lstm_ridge/` directory. SQL is the selected source; notebook 02 keeps its exports separate.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "config/sensors.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch from the repository root or notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
from config.a1_air_flow import CONFIG as A1_CONFIG
from config.a2_air_flow import CONFIG as A2_CONFIG
MODEL_CONFIGS = {cfg["name"]: dict(cfg) for cfg in (A1_CONFIG,A2_CONFIG)}
from importlib import reload
import src.training.preparation_experiments as preparation_experiments
reload(preparation_experiments)


## Verify workbook-selected inputs

Retain the checked feature audit and current-target roles from the original notebook.


In [ ]:
from zipfile import ZipFile
import xml.etree.ElementTree as ET
from config.sensors import SENSORS

WORKBOOK_PATH = PROJECT_ROOT / "Sensor_Configurations.xlsx"
NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
with ZipFile(WORKBOOK_PATH) as archive:
    shared = []
    if "xl/sharedStrings.xml" in archive.namelist():
        shared = ["".join(item.itertext()) for item in
                  ET.fromstring(archive.read("xl/sharedStrings.xml")).findall("m:si", NS)]
    workbook_rows = []
    for path in archive.namelist():
        if not (path.startswith("xl/worksheets/sheet") and path.endswith(".xml")):
            continue
        rows = []
        for row in ET.fromstring(archive.read(path)).findall(".//m:sheetData/m:row", NS):
            values = {}
            for cell in row.findall("m:c", NS):
                column = "".join(char for char in cell.attrib["r"] if char.isalpha())
                value = cell.find("m:v", NS)
                inline = cell.find("m:is", NS)
                text = value.text if value is not None else "".join(inline.itertext()) if inline is not None else ""
                if cell.get("t") == "s":
                    text = shared[int(text)]
                values[column] = (text or "").strip()
            rows.append(values)
        if any("A1 Aeration Flow" in row.values() for row in rows):
            workbook_rows.extend(rows)
if not workbook_rows:
    raise ValueError("Cannot find aeration input selections in the workbook.")
headers = next(row for row in workbook_rows if "A1 Aeration Flow" in row.values())
feature_headers = {value: column for column, value in headers.items()}
sensor_headers = next(row for row in workbook_rows if "Full Tag Name" in row.values())
tag_column = next(column for column, value in sensor_headers.items() if value == "Full Tag Name")
alias_by_tag = {sensor["fullTagName"].casefold(): alias for alias, sensor in SENSORS.items()}
feature_audit_records = []
for cfg, heading in [(A1_CONFIG, "A1 Aeration Flow"), (A2_CONFIG, "A2 Aeration Flow")]:
    checked = []
    for row in workbook_rows:
        mark = row.get(feature_headers[heading], "")
        if not mark or mark == heading:
            continue
        tag = row.get(tag_column, "").casefold()
        if not tag:
            continue
        if mark != "✓":
            raise ValueError(f"Unknown workbook selection mark: {mark!r}")
        if tag not in alias_by_tag:
            raise ValueError(f"Unmapped checked workbook tag: {tag}")
        alias = alias_by_tag[tag]
        checked.append(alias)
        feature_audit_records.append({
            "model": cfg["name"], "checked_alias": alias, "full_tag": tag,
            "role": "workbook-checked input", "workbook_checked": True,
        })
    if len(checked) != len(set(checked)) or cfg["target"] in checked:
        raise ValueError("Checked external inputs must be unique and exclude the current aeration target.")
    expected = set(cfg["features"])
    if set(checked) != expected:
        raise ValueError(f"Workbook/config mismatch for {cfg['name']}: {set(checked) ^ expected}")
    feature_audit_records.append({
        "model": cfg["name"], "checked_alias": cfg["target"],
        "full_tag": SENSORS[cfg["target"]]["fullTagName"],
        "role": "additional current-airflow autoregressive input", "workbook_checked": False,
    })
    # Preserve workbook feature order; the split cell separately adds current airflow.
    MODEL_CONFIGS[cfg["name"]] = {
        **cfg, "features": [alias for alias in checked if alias != cfg["target"]],
    }
feature_audit = pd.DataFrame(feature_audit_records)
display(feature_audit)
print("Verified 9 A1 and 10 A2 workbook inputs; current airflow adds one input per model.")


## Run controlled preparation comparisons and forecasts

This cell performs validation screening, then fits the selected preparation on pre-test data. It saves every preparation's held-out forecasts for review, the validation ranking, all-algorithm comparisons under the selected preparation, observation-quality audits and reproducibility metadata. The shared source exports stay unchanged. Re-running replaces only this notebook's strategy artifacts.


In [ ]:
results, OUTPUT_DIR = preparation_experiments.run_group(MODEL_CONFIGS, PROJECT_ROOT, "aeration")


## Review selected strategies and measured-label accuracy

`all_observed` uses labels indicated as observed by the source mask. `exclude_suspected_stale` is a sensitivity calculation that omits labels after at least 60 consecutive identical observed minutes. Constant measured targets have undefined R². Neither sensitivity nor test comparison changes validation selection. Methanol errors use m³/hour and airflow errors use m³/min; these are forecasts of recorded flows, not operating prescriptions.


In [ ]:
for name, result in results.items():
    print(name, "validation-selected preparation:", result["selection"])
    display(result["ranking"].head(10))
    display(result["metrics"])
    display(result["forecasts"].head())

    print("XGBoost / LSTM / Ridge comparison (identical sequence inputs):")
    display(result["algorithm_comparison"]["metrics"])
    display(result["algorithm_comparison"]["forecasts"].head())


## Visualize held-out forecasts and training context

Blue shows the validation-selected model, orange persistence and black observed future targets. Red markers flag long repeated target readings for source review. All forecast origins and future target timestamps are restricted to April 23 or earlier. Each CSV preserves the forecast-origin and future target timestamps.


In [ ]:
preparation_experiments.plot_results(results, OUTPUT_DIR)
print(f"Saved comparisons, models, quality audits, feature matrices and forecasts in {OUTPUT_DIR}")


In [ ]:
feature_audit.to_csv(OUTPUT_DIR / "workbook_feature_audit.csv", index=False)
